# B-2 · 学习准备周：装环境 + Git 最小六命令

> 学前缓冲带第 1 周，排在 B00（命令行）之前。
> 本周不写任何代码，目标是：**把工具装好，把代码管起来**。
> 学完整门课你会发现，每周都在用这一周的东西。

## 学习目标

学完本 notebook，你应该能够：

1. 装好并验证三件套：**VS Code**（编辑器）、**Git**（版本管理）、**uv**（Python 环境）；
2. 说清楚仓库、提交（commit）、远程（remote）分别是什么；
3. 熟练使用最小六命令：`clone` / `status` / `add` / `commit` / `push` / `pull`；
4. 故意制造一次合并冲突，并看懂冲突标记、手动解决它；
5. 知道哪些文件不该提交（`.gitignore` 的作用）。

> 先修要求：会基本电脑操作（安装软件、解压、打开文件夹）即可。
> 本 notebook 的演示 cell 在本地 `runs/00_basic/git_demo/` 里搭建「假远程仓库」，
> 不联网也能完整演练 push/pull/冲突——原理与 GitHub 完全相同。

## 1. 安装三件套

| 工具 | 干什么用 | Windows | Linux (Ubuntu) |
|------|----------|---------|----------------|
| **VS Code** | 写代码的编辑器 | 官网下载安装包：<https://code.visualstudio.com> | `sudo apt install code` 或官网下载 .deb |
| **Git** | 代码版本管理 | 官网下载 Git for Windows：<https://git-scm.com/download/win>（安装时一路默认即可） | `sudo apt install git` |
| **uv** | Python 依赖与虚拟环境管理 | PowerShell 执行：`irm https://astral.sh/uv/install.ps1 \| iex` | `curl -LsSf https://astral.sh/uv/install.sh \| sh` |

> 编辑器不是必须的——记事本也能写代码——但语法高亮和报错提示能救你无数次。
> 如果你的网络访问国外站点慢，uv 的安装也有国内镜像，详见 `docs/phase2_motrixlab.md` 等环境文档。

In [1]:
# 验证三件套：三条命令都能打印版本号，说明装好了
# （在 Windows 的 CMD/PowerShell 里敲同样的命令验证）
!git --version
!uv --version
!python --version

git version 2.34.1


uv 0.11.11 (x86_64-unknown-linux-gnu)


Python 3.11.15


## 2. Git 心智模型：三个比喻就够了

- **仓库（repository）= 带完整历史记录的文件夹**。表面上看就是普通文件夹，
  只是多了一个隐藏的 `.git/` 子目录，里面记着这个文件夹的「前世今生」。
- **提交（commit）= 给整个文件夹拍一张快照**，并附上一句说明（「改了什么、为什么」）。
  快照可以无限次回滚——这就是你敢大胆改代码的底气。
- **远程（remote）= 放在服务器（如 GitHub）上的另一份仓库**。
  `push` 把你的快照上传，`pull` 把别人的快照拉下来。

类比实验记录本：每天实验结束抄一页存档（commit），定期把副本寄到所里存档（push），
同门的存档你也可以要回来抄进自己的本子（pull）。**两个人抄了同一页的同一行**，
寄回去时就会打架——这就是「冲突」，本周最后会亲手制造并解决一次。

## 3. 先自报家门：配置用户名和邮箱

Git 要求每次提交都署名。第一次使用前配置一次（`--global` 表示对所有仓库生效）：

```bash
git config --global user.name "你的名字"
git config --global user.email "you@example.com"
```

下面的演示在每个仓库里单独配置（`git config` 不带 `--global`），效果一样，只是作用范围不同。

In [2]:
# 搭建本周演练场（本课程约定：运行产物都放 runs/ 下）
import os
import shutil
from pathlib import Path

NB_DIR = Path(os.getcwd())
PROJECT_ROOT = NB_DIR.parent.parent
DEMO = PROJECT_ROOT / "runs" / "00_basic" / "git_demo"
if DEMO.exists():
    shutil.rmtree(DEMO)          # 重跑本 notebook 时先清空，保证可重复
DEMO.mkdir(parents=True)
print("演练场:", DEMO)

演练场: /data/wangf/robot_rl_learn/runs/00_basic/git_demo


## 4. 六命令实操（上）：init / status / add / commit

在本地建一个仓库并完成第一次提交：

```bash
git init myrepo        # 把 myrepo/ 变成仓库（创建 .git/）
git status             # 看看现在什么状态：哪些文件变了、哪些已暂存
git add notes.md       # 把文件放进「暂存区」——准备拍进下一张快照
git commit -m "说明"   # 拍快照，-m 后面是这次的说明
```

**`add` 和 `commit` 为什么是两步？** 类比寄快递：`add` 是把东西放进箱子，
`commit` 是封箱贴面单。你可能改了 5 个文件但只想先寄 2 个——暂存区让你自由选择。

In [3]:
!git init -b main {DEMO}/myrepo
!git -C {DEMO}/myrepo config user.name "学员甲"
!git -C {DEMO}/myrepo config user.email "a@example.com"

Initialized empty Git repository in /data/wangf/robot_rl_learn/runs/00_basic/git_demo/myrepo/.git/


In [4]:
# 在仓库里建一个文件，然后走一遍 status → add → commit → log
(DEMO / "myrepo" / "notes.md").write_text("# 实验记录\n第一次提交。\n", encoding="utf-8")
!git -C {DEMO}/myrepo status --short
print("-" * 40)
!git -C {DEMO}/myrepo add notes.md
!git -C {DEMO}/myrepo commit -m "第一次提交：建立实验记录"
print("-" * 40)
!git -C {DEMO}/myrepo log --oneline

?? notes.md


----------------------------------------


[main (root-commit) 458d391] 第一次提交：建立实验记录
 1 file changed, 2 insertions(+)
 create mode 100644 notes.md


----------------------------------------
458d391 (HEAD -> main) 第一次提交：建立实验记录


`status --short` 输出里 `??` 表示「新文件，Git 还没在跟踪它」。
`log --oneline` 每行一个提交：前面是提交编号（哈希值前 7 位），后面是说明。

## 5. 六命令实操（下）：clone / push / pull

真实的「远程」在 GitHub 上，但原理不依赖 GitHub——**任何一份仓库都可以当远程**。
下面在本地建一个「裸仓库」（bare，只存历史不存工作文件，服务器上就是这种）充当远程：

In [5]:
# 建一个假远程，把 myrepo 推上去，再克隆出第二份（模拟「同门」）
!git init --bare -b main {DEMO}/remote.git
!git -C {DEMO}/myrepo remote add origin {DEMO}/remote.git
!git -C {DEMO}/myrepo push -u origin main
print("-" * 40)
!git clone {DEMO}/remote.git {DEMO}/clone_b
!git -C {DEMO}/clone_b config user.name "学员乙"
!git -C {DEMO}/clone_b config user.email "b@example.com"
!git -C {DEMO}/clone_b config pull.rebase false
!git -C {DEMO}/myrepo config pull.rebase false

Initialized empty Git repository in /data/wangf/robot_rl_learn/runs/00_basic/git_demo/remote.git/


Enumerating objects: 3, done.
Counting objects: 100% (3/3), done.
Writing objects: 100% (3/3), 284 bytes | 284.00 KiB/s, done.
Total 3 (delta 0), reused 0 (delta 0), pack-reused 0
To /data/wangf/robot_rl_learn/runs/00_basic/git_demo/remote.git
 * [new branch]      main -> main
Branch 'main' set up to track remote branch 'main' from 'origin'.


----------------------------------------
Cloning into '/data/wangf/robot_rl_learn/runs/00_basic/git_demo/clone_b'...
done.


In [6]:
# 学员乙在自己的克隆里改文件 → 提交 → push 到远程
(DEMO / "clone_b" / "notes.md").write_text(
    "# 实验记录\n第一次提交。\n乙补充：W01 已完成。\n", encoding="utf-8"
)
!git -C {DEMO}/clone_b add notes.md
!git -C {DEMO}/clone_b commit -m "乙：补充 W01 进度"
!git -C {DEMO}/clone_b push

[main 75808b2] 乙：补充 W01 进度
 1 file changed, 1 insertion(+)


Enumerating objects: 5, done.
Counting objects: 100% (5/5), done.
Delta compression using up to 24 threads
Compressing objects: 100% (2/2), done.
Writing objects: 100% (3/3), 334 bytes | 334.00 KiB/s, done.
Total 3 (delta 0), reused 0 (delta 0), pack-reused 0
To /data/wangf/robot_rl_learn/runs/00_basic/git_demo/remote.git
   458d391..75808b2  main -> main


In [7]:
# 学员甲 pull 拉取乙的工作
!git -C {DEMO}/myrepo pull
!cat {DEMO}/myrepo/notes.md

remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 3 (delta 0), reused 0 (delta 0), pack-reused 0
Unpacking objects: 100% (3/3), 314 bytes | 314.00 KiB/s, done.
From /data/wangf/robot_rl_learn/runs/00_basic/git_demo/remote
   458d391..75808b2  main       -> origin/main
Updating 458d391..75808b2
Fast-forward
 notes.md | 1 +
 1 file changed, 1 insertion(+)


# 实验记录
第一次提交。
乙补充：W01 已完成。


到这一步，日常 90% 的 Git 操作你就见过了。真实场景只是把 `remote.git` 换成
`https://github.com/ten2net/robot_rl_learn.git` 这样的地址。

## 6. 故意制造一次冲突（本周最重要的一节）

**冲突是怎么来的**：甲和乙都基于同一个版本，改了**同一行**，先后推送。
先推的甲顺利上去；乙再推时被拒绝——因为远程已经有了乙没见过的新提交。

In [8]:
# 甲改了 notes.md 第二行并推送成功
(DEMO / "myrepo" / "notes.md").write_text(
    "# 实验记录\n甲的修改：第一次提交完成。\n乙补充：W01 已完成。\n", encoding="utf-8"
)
!git -C {DEMO}/myrepo add notes.md
!git -C {DEMO}/myrepo commit -m "甲：改写第二行"
!git -C {DEMO}/myrepo push

[main 4111c66] 甲：改写第二行
 1 file changed, 1 insertion(+), 1 deletion(-)


Enumerating objects: 5, done.
Counting objects: 100% (5/5), done.
Delta compression using up to 24 threads
Compressing objects: 100% (2/2), done.
Writing objects: 100% (3/3), 345 bytes | 345.00 KiB/s, done.
Total 3 (delta 0), reused 0 (delta 0), pack-reused 0
To /data/wangf/robot_rl_learn/runs/00_basic/git_demo/remote.git
   75808b2..4111c66  main -> main


In [9]:
# 乙也改了同一行（基于旧版本），提交后推送——被拒绝
(DEMO / "clone_b" / "notes.md").write_text(
    "# 实验记录\n乙的修改：第一次提交完成。\n乙补充：W01 已完成。\n", encoding="utf-8"
)
!git -C {DEMO}/clone_b add notes.md
!git -C {DEMO}/clone_b commit -m "乙：改写第二行"
!git -C {DEMO}/clone_b push

[main 6db67f9] 乙：改写第二行
 1 file changed, 1 insertion(+), 1 deletion(-)


To /data/wangf/robot_rl_learn/runs/00_basic/git_demo/remote.git
 ! [rejected]        main -> main (fetch first)
error: failed to push some refs to '/data/wangf/robot_rl_learn/runs/00_basic/git_demo/remote.git'
hint: Updates were rejected because the remote contains work that you do
hint: not have locally. This is usually caused by another repository pushing
hint: to the same ref. You may want to first integrate the remote changes
hint: (e.g., 'git pull ...') before pushing again.
hint: See the 'Note about fast-forwards' in 'git push --help' for details.


看到 `rejected` / `non-fast-forward` 不要慌，它的意思很直白：
「远程有你没见过的提交，先 `pull` 合并再推」。照做：

In [10]:
# 乙 pull：Git 尝试自动合并，但同一行被两边都改了——合并失败，标记冲突
!git -C {DEMO}/clone_b pull
print("-" * 40)
!cat {DEMO}/clone_b/notes.md

remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 3 (delta 0), reused 0 (delta 0), pack-reused 0
Unpacking objects: 100% (3/3), 325 bytes | 325.00 KiB/s, done.
From /data/wangf/robot_rl_learn/runs/00_basic/git_demo/remote
   75808b2..4111c66  main       -> origin/main
Auto-merging notes.md
CONFLICT (content): Merge conflict in notes.md
Automatic merge failed; fix conflicts and then commit the result.


----------------------------------------
# 实验记录
<<<<<<< HEAD
乙的修改：第一次提交完成。
甲的修改：第一次提交完成。
>>>>>>> 4111c66d75fed4d6eacfa3f607e64c77bb2d0019
乙补充：W01 已完成。


打开冲突文件，你会看到这样的**冲突标记**：

```
<<<<<<< HEAD
乙的修改：第一次提交完成。
=======
甲的修改：第一次提交完成。
>>>>>>> 甲的提交编号
```

读法：

- `<<<<<<< HEAD` 到 `=======` 之间：**你这边**的内容（HEAD = 你当前的提交）；
- `=======` 到 `>>>>>>>` 之间：**对方（远程拉下来的）**的内容；
- 解决方式：**手动把这一段编辑成你想要的样子**（留一边、留另一边、或两边融合），
  并把 `<<<<<<<` `=======` `>>>>>>>` 三行标记**全部删掉**，然后 `add` + `commit` + `push`。

In [11]:
# 乙手动解决：融合双方内容，删掉标记，再提交推送
(DEMO / "clone_b" / "notes.md").write_text(
    "# 实验记录\n甲乙合并：第一次提交完成。\n乙补充：W01 已完成。\n", encoding="utf-8"
)
!git -C {DEMO}/clone_b add notes.md
!git -C {DEMO}/clone_b commit -m "解决冲突：合并甲乙的第二行修改"
!git -C {DEMO}/clone_b push
print("-" * 40)
!git -C {DEMO}/clone_b log --oneline

[main e0ff199] 解决冲突：合并甲乙的第二行修改


Enumerating objects: 10, done.
Counting objects: 100% (10/10), done.
Delta compression using up to 24 threads
Compressing objects: 100% (4/4), done.
Writing objects: 100% (6/6), 716 bytes | 716.00 KiB/s, done.
Total 6 (delta 0), reused 0 (delta 0), pack-reused 0
To /data/wangf/robot_rl_learn/runs/00_basic/git_demo/remote.git
   4111c66..e0ff199  main -> main


----------------------------------------
e0ff199 (HEAD -> main, origin/main, origin/HEAD) 解决冲突：合并甲乙的第二行修改
6db67f9 乙：改写第二行
4111c66 甲：改写第二行
75808b2 乙：补充 W01 进度
458d391 第一次提交：建立实验记录


**防冲突的最好办法不是技术，是习惯**：动手改代码前先 `git pull`，
提交频率高一些（小步提交），冲突自然又少又小。

## 7. 什么不该提交：`.gitignore`

仓库根目录的 `.gitignore` 文件列出一批「Git 不要跟踪」的路径。本项目的约定
（见 `.gitignore` 与 `AGENTS.md`）：

- `runs/`、`logs/`、`checkpoints/`：训练产物，体积大、可再生，不提交；
- `.venv/`：虚拟环境，每个人本地自己 `uv sync` 生成，不提交；
- `__pycache__/`、`.pytest_cache/`：自动生成的缓存，不提交。

> 判断标准一句话：**别人 clone 后能自己生成的东西，都不提交**。
> 提交前 `git status` 看一眼清单，发现误加了不该提交的文件，用
> `git rm --cached 文件路径` 把它移出跟踪（文件本身不删）。

## 8. 在本课程中你会怎么用 Git

1. 克隆课程仓库：`git clone https://github.com/ten2net/robot_rl_learn.git`；
2. 每周学习成果（journal 小结、练习代码）`add` + `commit` + `push`，形成学习轨迹；
3. 课程更新了内容，`git pull` 拉取最新版——如果和你本地的修改冲突，
   用第 6 节的方法解决；
4. 本 notebook 的演示产物在 `runs/00_basic/git_demo/`，已被 gitignore，删了重跑即可。

## ✏️ 练习

在**你自己的终端**里完成（命令 Windows 与 Linux 完全一致——这正是 Git 的优点）。
建议新建一个临时目录（如 `runs/00_basic/git_practice/`）当练习场。

**练习 1（20 分，难度 ★）安装验证**
在终端依次运行 `git --version`、`uv --version`、`python --version`，
三条都打印出版本号即通过。如果某条提示「command not found / 不是内部或外部命令」，
回到第 1 节检查安装（多半是 PATH 问题，重开一个终端窗口再试）。

**练习 2（25 分，难度 ★★）第一个仓库**
新建目录 `my_first_repo`，初始化为 Git 仓库，创建一个 `hello.txt`，
做**三次**提交（每次改一点内容），最后用 `git log --oneline` 展示三行提交记录。

**练习 3（25 分，难度 ★★）模拟远程协作**
建一个裸仓库 `origin.git` 当远程，把你的仓库推上去；
再 `git clone` 出第二份工作副本，在副本里改文件、提交、push，
回到第一个仓库 `pull`，确认改动同步过来了。

**练习 4（30 分，难度 ★★★）冲突实战**
重复第 6 节的完整流程：两份副本改同一行 → 先后推送 → 解决冲突 → 推送成功。
然后用 3–4 句话向一个完全没用过 Git 的人解释：
`<<<<<<<`、`=======`、`>>>>>>>` 三行标记各是什么意思？

<details>
<summary>👉 参考答案（先独立做，再点开）</summary>

**练习 1**

```bash
git --version      # 例如 git version 2.34.1
uv --version       # 例如 uv 0.x.x
python --version   # 例如 Python 3.11.x
```

Windows 下新装的命令如果找不到，**关掉终端重新打开**（PATH 要新窗口才刷新）。

**练习 2**

```bash
mkdir my_first_repo && cd my_first_repo
git init -b main
git config user.name "你的名字" && git config user.email "you@example.com"
echo "v1" > hello.txt && git add hello.txt && git commit -m "第一次提交"
echo "v2" >> hello.txt && git add hello.txt && git commit -m "第二次提交"
echo "v3" >> hello.txt && git add hello.txt && git commit -m "第三次提交"
git log --oneline    # 应看到 3 行，最新的在最上面
```

**练习 3**

```bash
git init --bare -b main ../origin.git
git remote add origin ../origin.git
git push -u origin main
cd .. && git clone origin.git work_copy && cd work_copy
git config user.name "乙" && git config user.email "b@example.com"
git config pull.rebase false
echo "from copy" >> hello.txt && git add hello.txt
git commit -m "副本的修改" && git push
cd ../my_first_repo && git config pull.rebase false && git pull
cat hello.txt        # 应包含 from copy
```

**练习 4**

流程照第 6 节抄即可。三行标记的解释参考说法：
`<<<<<<< HEAD` 到 `=======` 之间是**我自己当前版本**的内容；
`=======` 到 `>>>>>>>` 之间是**对方（拉下来的）版本**的内容；
Git 无法判断同一行该听谁的，就把两个版本都摆出来，
由人手动编辑成最终样子并删掉这三行标记，再提交。
</details>

## 延伸阅读

- [Pro Git 中文版（官方免费书）](https://git.progit.cn/)——第 1–3 章覆盖本周全部内容，值得收藏；
- [Learn Git Branching（可视化交互练习）](https://learngitbranching.js.org/?locale=zh_CN)——把分支操作做成小游戏，玩前 5 关；
- [GitHub 官方 Hello World 教程](https://docs.github.com/zh/get-started/start-your-journey/hello-world)——注册账号后跟着走一遍；
- [gitignore 模板库](https://github.com/github/gitignore)——各种语言的标准 .gitignore 写法。

## 小结

- 三件套装好用三条 `--version` 验证；**找不到命令先重开终端窗口**；
- 六命令口诀：**克隆 clone、看状态 status、装箱 add、封箱 commit、上传 push、同步 pull**；
- 冲突不是事故，是 Git 说「同一行两个人改了，你定」——看懂 `<<<<<<<` 标记手动合并即可；
- 别人能自己生成的东西（虚拟环境、训练产物、缓存）不提交，交给 `.gitignore`；
- 下一周 B00 学命令行——本周你已经用了不少，会发现它们突然变得亲切了。